# NORTHSTAR practical — is the data any good?

**DASC513 · Data Engineering block · Week 7 · data quality**

> ### Before you do anything else
> **File → Save a copy in Drive.** Until you do that you are looking at a read-only
> copy and your work disappears when the tab closes. Do it now, then carry on.

This notebook runs in Google Colab, or on your own machine if you have Python and
pandas. Nothing needs installing. The data loads in the next cell straight from
the course GitHub repository; if that fails (no internet, or you are working
offline), it will ask you to upload the CSV files you downloaded from Canvas.

## How to use this notebook

A notebook is a sequence of **cells**. Grey cells are code; white cells are
writing. Run a cell by clicking into it and pressing **Shift + Enter**.

Three kinds of cell in this notebook:

| Marker | What to do |
|---|---|
| **READ** | Explanation. No code to run. |
| **RUN** | Code that is already written. Read it first, predict what it will do, then run it. |
| **YOUR TURN** | Code with something missing, or a written answer to give. |

Two rules that will save you an hour of confusion:

1. **Cells run in the order you run them, not the order they appear.** If
   something breaks unexpectedly, go to *Kernel → Restart and Run All* and start
   again from the top.
2. **Read code before running it.** Say it out loud in English. `df["sex"]`
   is "the sex column of df". `df["sex"].value_counts()` is "count how many
   times each value appears in the sex column of df". If you can say it, you
   understand it.

## The enquiry loop

Every task below follows the same five steps. This is the actual method; the
pandas is just how you carry it out.

1. **Observe** — look at the data without deciding anything yet.
2. **Question** — turn what you noticed into a question that has an answer.
3. **Test** — write the smallest piece of code that answers it.
4. **Interpret** — what does the answer mean *for the research question*?
5. **Decide** — fix it, flag it, exclude it, or accept it. Write down which and why.

Step 5 is the one people skip. A data quality report full of tables and empty of
decisions is not a data quality report.

---
# Part 0 — Opening the data

**READ.** `pandas` is the library everyone uses for table-shaped data in Python. By convention it is imported as `pd`. A table in pandas is called a **DataFrame**, and almost everyone calls their first one `df`.

In [ ]:
# RUN. Load the data. Works in Colab, and on your own machine if the CSV files
# are beside the notebook. If neither, Colab will ask you to upload them.
import io, os
import pandas as pd

DATA_REPO = "https://raw.githubusercontent.com/civic-health-innovation-labs/dasc513-data-engineering/main/"
DATA_URL = DATA_REPO + "NORTHSTAR_redcap_export.csv"
DICT_URL = DATA_REPO + "NORTHSTAR_data_dictionary.csv"

_uploaded = {}

def load(name, url):
    if url:
        try:
            return pd.read_csv(url, dtype=str)
        except Exception as e:          # offline, or the repository moved
            print(f"Could not fetch {name} from GitHub ({type(e).__name__}); trying a local copy.")
    if os.path.exists(name):
        return pd.read_csv(name, dtype=str)
    if name not in _uploaded:
        try:
            from google.colab import files
        except ImportError:
            raise FileNotFoundError(f"{name} not found. Put it next to this notebook, or check the URL above.")
        print(f"Choose {name} (from Canvas) when the button appears.")
        _uploaded.update(files.upload())
    return pd.read_csv(io.BytesIO(_uploaded[name]), dtype=str)

df = load("NORTHSTAR_redcap_export.csv", DATA_URL)
data_dictionary = load("NORTHSTAR_data_dictionary.csv", DICT_URL)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print(f"loaded {df.shape[0]} records with {df.shape[1]} fields, and a data dictionary of {len(data_dictionary)} rows")

**READ — why `dtype=str`?**

Left alone, pandas guesses what each column contains. It is quite good at
guessing and that is exactly the problem: it will quietly turn `"3.7"` into the
number 3.7 and `"01"` into the number 1, and it will guess wrong on at least one
of the date columns in this file.

Today we want to see the data **as it was actually entered**, so we read
everything as text and convert later, deliberately, when we have decided what
each column should be.

> **Challenge to think about (discussion, not code):** what is the risk of the
> opposite habit — always letting the tool convert types for you? Where would
> that bite in a real pipeline?

In [ ]:
# RUN. head() shows the first few rows. The .T flips rows and columns, which is
# much easier to read when a table is this wide.
df.head(3).T

In [ ]:
# RUN. The data dictionary is a table too. This shows the first few rows.
# Later you will want:  data_dictionary[data_dictionary["variable_name"] == "some_field"]
data_dictionary.head(8)

**YOUR TURN 1.**

Look at the output above and answer in the cell below (in words, not code):

- What does one **row** represent?
- Name three columns that could identify a real person.
- Name one column you do not understand at all.

Then check your third answer against `NORTHSTAR_data_dictionary.csv`. If it is
not in there, that is not your mistake — write down that the dictionary is
incomplete, because that is itself a finding.

*Your answer here (double-click to edit this cell):*

- A row represents…
- Possible identifiers:
- I do not understand:

---
# Part 1 — Asking a single column a question

**READ.** The single most useful thing you can do to a column is count what is
in it. `value_counts()` does that.

In [ ]:
# RUN. Predict the answer before you run it: how many different values
# do you expect the sex column to contain?
df["sex"].value_counts(dropna=False)

**Interpret.** You probably expected two values. Count them.

This is a **conformance** problem: the values do not conform to the format the
data dictionary specifies. It happens because the CRF used a free-text box where
it should have used a dropdown, and because three sites had different research
nurses typing into it.

**Why it matters:** any code that does `df[df["sex"] == "F"]` silently drops
patients. Not with an error. Silently.

In [ ]:
# YOUR TURN 2. Pick a different column and count its values.
# Try: "consent_type", "outcome", "smoking_status", "ethnic_category", "site_code"
# Change the column name below and run it.

df["consent_type"].value_counts(dropna=False)

**YOUR TURN 2, part b.** Run the cell above on `ethnic_category`. You will see
both `99` and `999`.

One of those is a real code meaning "not known" (it is in the NHS Data
Dictionary). The other is somebody's shorthand for "I didn't fill this in".

- Which is which? How would you find out?
- What happens to your analysis if you treat them as the same thing? What
  happens if you treat `99` as a real ethnic group?

---
# Part 2 — Missing data is not one thing

**READ.** "Missing" in this file appears in at least five disguises: an empty
cell, the text `NULL`, the text `N/A`, the number `999`, and the word `Unknown`.
Different people, different sites, different days.

pandas only recognises the truly empty ones. So the first job is to write down
what *we* mean by missing, and then apply that consistently.

In [ ]:
# RUN. A list of the markers that mean "no value" in this file.
MISSING_MARKERS = ["", "NULL", "N/A", "999", "Unknown", "unknown", "NA"]

def is_missing(column):
    # True/False for every cell: is this one of our missing markers?
    return column.isna() | column.isin(MISSING_MARKERS)

# Try it on one column
is_missing(df["ferritin_adm"]).sum()      # .sum() counts the Trues

In [ ]:
# RUN. Missingness across a set of columns we care about.
columns_of_interest = [
    "crp_adm", "wcc_adm", "lymph_adm", "creat_adm", "urea_adm",
    "ddimer_adm", "ferritin_adm", "adm_weight_kg", "adm_height_cm",
    "symptom_onset_date", "ethnic_category",
]

missing_table = pd.DataFrame({
    "n_missing": [is_missing(df[c]).sum() for c in columns_of_interest],
    "pct_missing": [round(100 * is_missing(df[c]).mean(), 1) for c in columns_of_interest],
}, index=columns_of_interest).sort_values("pct_missing", ascending=False)

missing_table

**Interpret.** Look at the worst offenders. `ddimer_adm` and `ferritin_adm` are
the two markers most associated with severity in respiratory infection, and they
are the two least complete.

**Now stop.** Before you decide what to do about it, ask the question that
separates a data engineer from someone running a profiling tool:

> **Is the missingness spread evenly across the dataset, or is it concentrated?**

An overall percentage cannot tell you. Part 3 can.

---
# Part 3 — The question behind the question

**READ.** `groupby` splits the table into groups and does the same calculation to
each one. It is the most important verb in this notebook.

Here we split by site and recalculate the missingness.

In [ ]:
# RUN. Read this in English first:
# "for each site, for each of these columns, what fraction is missing?"

by_site = df.copy()
for c in columns_of_interest:
    by_site[c] = is_missing(by_site[c])

(by_site
    .groupby("redcap_data_access_group")[columns_of_interest]
    .mean()
    .round(2)
    .T)                     # .T so columns run down the page

**YOUR TURN 3 — the important one.**

Look at the `wirral_park` column and compare it with the other two.

Answer these in the cell below, in words:

1. What is the pattern? State it in one sentence.
2. Give a plausible clinical or operational explanation. (Wirral Park is a
   smaller district general hospital.)
3. Someone proposes to train a model using D-dimer as a predictor, dropping any
   patient with a missing value. **Who ends up in the training set?** What has
   just happened to your study population?
4. Is this missing completely at random, missing at random, or missing not at
   random? Can you tell from the data alone?

*Your answer here:*

1.
2.
3.
4.

---
# Part 4 — Conformance: three sites, three date formats

**READ.** Run the cell below and look at `adm_date` for each site.

In [ ]:
# RUN
df.groupby("redcap_data_access_group")["adm_date"].head(2)

In [ ]:
# RUN. Same thing, laid out so the pattern is obvious.
df[["record_id", "redcap_data_access_group", "adm_date", "dob", "outcome_date"]].head(12)

**Interpret.** Three sites, three conventions:

| Site | Format | Example |
|---|---|---|
| Mersey General | `DD/MM/YYYY` | `16/12/2024` |
| Calderbrook | `YYYY-MM-DD` | `2024-10-11` |
| Wirral Park | `DD-Mon-YY` | `23-Jan-25` |

**Why this is dangerous rather than merely annoying:** `05/06/2024` is a valid
date in two of these conventions and means two different things. A parser that
guesses will not warn you. It will just be wrong for some rows.

In [ ]:
# RUN. Parse each site's dates using that site's format.
SITE_FORMATS = {
    "mersey_general": "%d/%m/%Y",
    "calderbrook":    "%Y-%m-%d",
    "wirral_park":    "%d-%b-%y",
}

def parse_dates(frame, column):
    # Parse a date column, using the right format for each site.
    out = pd.Series(pd.NaT, index=frame.index)
    for site, fmt in SITE_FORMATS.items():
        rows = frame["redcap_data_access_group"] == site
        out[rows] = pd.to_datetime(frame.loc[rows, column], format=fmt, errors="coerce")
    return out

df["adm_date_parsed"] = parse_dates(df, "adm_date")

# How many failed to parse?
failed = df["adm_date_parsed"].isna()
print("failed to parse:", failed.sum())
df.loc[failed, ["record_id", "redcap_data_access_group", "adm_date"]]

**YOUR TURN 4.**

Some rows failed. Look at them.

- Three of them failed because of their format. Compare each to its site's
  convention. What happened?
- The fourth (record 99) is different. Look at that whole record —
  `df[df["record_id"] == "99"]` — and say what is going on with it.
- What does this tell you about the assumption "each site is internally
  consistent"?
- `errors="coerce"` means "if you cannot parse it, put a blank". What would have
  happened if we had not written that? Is silently blanking them a good idea?

---
# Part 5 — Plausibility: could this value be true?

**READ.** Conformance asks "is it the right shape". Plausibility asks "could a
human being actually have this value". A temperature of `3.7` is a perfectly
well-formed number.

In [ ]:
# RUN. Convert some clinical columns to numbers so we can compare them.
numeric_columns = ["temp_adm", "hr_adm", "rr_adm", "sbp_adm", "spo2_adm",
                   "adm_weight_kg", "adm_height_cm", "creat_adm", "hosp_los_days"]

for c in numeric_columns:
    df[c + "_num"] = pd.to_numeric(df[c], errors="coerce")

# Ranges a clinician would accept. These are the interesting decisions.
PLAUSIBLE = {
    "temp_adm":      (34.0, 42.0),
    "hr_adm":        (30, 200),
    "rr_adm":        (6, 60),
    "sbp_adm":       (50, 250),
    "spo2_adm":      (50, 100),
    "adm_weight_kg": (30, 300),
    "adm_height_cm": (120, 220),
    "hosp_los_days": (0, 200),
}

problems = []
for column, (low, high) in PLAUSIBLE.items():
    values = df[column + "_num"]
    bad = values.notna() & ((values < low) | (values > high))
    for rid, v in zip(df.loc[bad, "record_id"], values[bad]):
        problems.append({"record_id": rid, "field": column, "value": v,
                         "expected": f"{low} to {high}"})

pd.DataFrame(problems)

**YOUR TURN 5.**

For each row in that table, decide what you would actually do, and why:

| Record | Field | Value | Your decision |
|---|---|---|---|
| | | | correct it / set to missing / exclude the patient / query with the site |

Some of these are guessable (a temperature of `3.7` is almost certainly `37`).
Some are not. **Which ones are you allowed to guess?**

Three of the rows are not data errors at all. Look for the value `999`, and think
about what we said in Part 2. What has gone wrong in the *code*, not the data?

One more is genuinely ambiguous: a respiratory rate of 5 is very low, but it is
not impossible in a patient who is about to arrest. Would you flag it, correct
it, or leave it? Who decides?

In [ ]:
# YOUR TURN 5b. Write one cross-field check. Here is the skeleton.
df["outcome_date_parsed"] = parse_dates(df, "outcome_date")

impossible = df["outcome_date_parsed"] < df["adm_date_parsed"]

df.loc[impossible, ["record_id", "adm_date", "outcome_date", "hosp_los_days"]]

Now write your own in the empty cell below. Ideas:

- length of stay does not match `outcome_date` minus `adm_date`
- a patient with `icu_admit = Yes` but no `icu_admit_date`
- a patient who died but has a discharge destination of "Home"
- symptom onset after admission

Pick one. You are looking for rows where two fields disagree.

In [ ]:
# YOUR TURN 5b — your own cross-field check here.

---
# Part 6 — Is anyone in here twice?

**READ.** Deduplication sounds mechanical. It is not: it is a decision about
whether two records are the same person, made with imperfect information. This is
the same problem as **record linkage**, just within one file.

In [ ]:
# RUN. Attempt 1: same name and same date of birth.
name_dob = df.duplicated(subset=["patient_forename", "patient_surname", "dob"], keep=False)
print("matches on name + date of birth:", name_dob.sum())
df.loc[name_dob, ["record_id", "patient_forename", "patient_surname", "dob"]]

In [ ]:
# RUN. Attempt 2: same NHS number.
nhs_dup = df.duplicated(subset=["nhs_number"], keep=False) & (df["nhs_number"] != "")
print("matches on NHS number:", nhs_dup.sum())
df.loc[nhs_dup, ["record_id", "redcap_data_access_group", "patient_forename",
                 "patient_surname", "nhs_number", "dob", "postcode",
                 "hospital_number", "adm_date"]]

**YOUR TURN 6.**

The first attempt found nothing. The second found a pair.

1. Look at the two records carefully. List **every** field where they differ.
2. Why did matching on name and date of birth fail?
3. Are these the same person admitted twice, or the same admission entered twice
   at two sites? Does it matter? What would you need to know to decide?
4. If you delete one, which one, and what do you lose?

---
# Part 7 — Your verdict

**YOUR TURN 7.** In the cell below, write six to ten sentences answering:

> A clinical colleague wants to start training a model on this dataset next
> week, using the full set of admission observations and blood results, to
> predict death in hospital. **What do you tell them?**

Structure it as: what you found, what you would fix, what you would refuse, and
what you would need from someone else before proceeding.

You are not being asked to say no. You are being asked to say what has to be true
first.

*Your answer here:*